"""
preprocess.py

This module preprocesses the extracted landmark dataset before training.

Responsibilities:
- Load the landmark CSV file.
- Normalize landmark coordinates.
- Generate additional geometric features (optional).
- Split the dataset into training, validation, and testing sets.
- Save the processed datasets for model training.

The purpose of this module is to improve model robustness and ensure
consistent input features.
"""

In [20]:
import config
import pandas as pd


In [21]:
df = pd.read_csv(config.LANDMARKS_CSV)

In [22]:
print(df.shape)
print(df.head(3))
print(df.dtypes)

(140828, 64)
  label        x0        y0            z0        x1        y1        z1  \
0     A  0.465325  0.708383 -5.010667e-07  0.555823  0.661244 -0.029832   
1     A  0.360139  0.952337 -5.804150e-07  0.447511  0.914585 -0.024304   
2     A  0.428328  0.523921 -2.575191e-07  0.484031  0.516531 -0.018898   

         x2        y2        z2  ...       z17       x18       y18       z18  \
0  0.611590  0.556126 -0.036495  ... -0.029435  0.420653  0.517107 -0.068952   
1  0.508482  0.819816 -0.028193  ... -0.017192  0.325507  0.751496 -0.055004   
2  0.532599  0.455391 -0.020857  ... -0.003997  0.438966  0.360599 -0.026977   

        x19       y19       z19       x20       y20       z20  
0  0.438396  0.581985 -0.067245  0.448647  0.623101 -0.052515  
1  0.335612  0.815825 -0.047659  0.346568  0.858426 -0.029108  
2  0.432422  0.408642 -0.023362  0.426960  0.443932 -0.012639  

[3 rows x 64 columns]
label     object
x0       float64
y0       float64
z0       float64
x1       float64
 

In [23]:
dublicates=df.duplicated().sum()
missings=df.isnull().sum()
print(dublicates,missings)

59913 label    0
x0       0
y0       0
z0       0
x1       0
        ..
y19      0
z19      0
x20      0
y20      0
z20      0
Length: 64, dtype: int64


In [24]:
duplicate_rows = df[df.duplicated(keep=False)]

print(duplicate_rows.head(20))
print(df[df.duplicated()].head())

    label        x0        y0            z0        x1        y1        z1  \
503     A  0.456254  0.587702 -6.443583e-07  0.570339  0.504806 -0.035954   
504     A  0.485474  0.616646 -7.179940e-07  0.602393  0.541329 -0.028766   
505     A  0.719152  0.678725 -5.860258e-07  0.793566  0.623645 -0.030276   
506     A  0.711866  0.752371 -5.115813e-07  0.828867  0.652734 -0.042542   
507     A  0.716096  0.758583 -4.781011e-07  0.833443  0.666843 -0.041355   
508     A  0.723811  0.776854 -5.130469e-07  0.839596  0.677036 -0.041665   
509     A  0.718564  0.782512 -5.119534e-07  0.842921  0.685035 -0.040458   
510     A  0.723495  0.784671 -5.776235e-07  0.844440  0.690429 -0.046588   
511     A  0.716912  0.789708 -6.107376e-07  0.836023  0.705881 -0.044324   
512     A  0.711770  0.783728 -5.041232e-07  0.835397  0.706519 -0.049639   
513     A  0.708537  0.795938 -5.905972e-07  0.835191  0.715445 -0.044930   
514     A  0.702098  0.794575 -5.293936e-07  0.822628  0.724569 -0.045699   

whether the duplicate features have different labels

If We get only
1

then every duplicated feature vector belongs to exactly one label, which is good.

If you get
2
or more, then the same landmark coordinates are associated with different letters, which would be a data issue

In [25]:
duplicates = df[df.duplicated(subset=df.columns[1:], keep=False)]

print(
    duplicates.groupby(list(df.columns[1:]))["label"]
    .nunique()
    .value_counts()
)

label
1    59913
2       23
Name: count, dtype: int64


In [26]:
#Y=df["label"].unique()
Y=df["label"]
X = df.drop(columns=["label"])

print(X.shape)
print(Y.shape)

(140828, 63)
(140828,)


Load landmarks.csv
       
        │
        ▼

Check for missing values
        
        │
        ▼

Train/Test split (70/30)
        
        │
        ▼

Validation/Test split (50/50)
        
        │
        ▼

Save

train.csv
validation.csv
test.csv

In [27]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
Y_encoded=label_encoder.fit_transform(Y)
print(Y_encoded)

for index, label in enumerate(label_encoder.classes_):
    print(f"{label} -> {index}")

[ 0  0  0 ... 25 25 25]
A -> 0
B -> 1
C -> 2
D -> 3
E -> 4
F -> 5
G -> 6
H -> 7
I -> 8
J -> 9
K -> 10
L -> 11
M -> 12
N -> 13
O -> 14
P -> 15
Q -> 16
R -> 17
S -> 18
T -> 19
U -> 20
V -> 21
W -> 22
X -> 23
Y -> 24
Z -> 25
del -> 26
space -> 27


In [28]:
from sklearn.model_selection import train_test_split

# Step 1: Split the data to 70% Train and 30% (Testing+Validaton)
X_train, X_test, Y_train, Y_test=train_test_split(
    X,
    Y_encoded,
    test_size=0.3,
    stratify=Y_encoded, #will make the data distrubuted with the same % as in the dataset
    random_state=42 #shuffling randomly

)

# Step 2: Split the 30% Temp equally (0.50) into Validation and Test (15% each of total)
X_val, X_test, Y_val, Y_test = train_test_split(
    X_test, Y_test, test_size=0.50, stratify=Y_test, random_state=42
)

print(f"Train shapes:      X={X_train.shape}, Y={Y_train.shape}")
print(f"Validation shapes: X={X_val.shape}, Y={Y_val.shape}")
print(f"Test shapes:       X={X_test.shape}, Y={Y_test.shape}")

Train shapes:      X=(98579, 63), Y=(98579,)
Validation shapes: X=(21124, 63), Y=(21124,)
Test shapes:       X=(21125, 63), Y=(21125,)


In [ ]:
train_df=pd.DataFrame(X_train).copy()
train_df["label"]=Y_train

test_df=pd.DataFrame(X_test).copy()
test_df["label"]=Y_test

val_df=pd.DataFrame(X_val).copy()
val_df["label"]=Y_val

train_df.to_csv(config.TRAIN_CSV,index=False)
test_df.to_csv(config.TEST_CSV,index=False)
val_df.to_csv(config.VALIDATION_CSV,index=False)

print("All datasets successfully saved to CSV!")

In [ ]:

import numpy as np
# --- 1. OVERLAPPING SLIDING WINDOW (Fixes mixed-sign data concern) ---
sequence_length = 10
X_sequences = []
Y_sequences = []

# Convert to numpy for fast loop execution speeds
X_numpy = X.values
Y_numpy = Y_encoded

for i in range(0, len(X_numpy) - sequence_length + 1):
    window_labels = Y_numpy[i : i + sequence_length]
    
    # Check if ALL 10 frames belong to the same ASL sign (removes mixed transitions)
    if np.all(window_labels == window_labels[0]):
        X_sequences.append(X_numpy[i : i + sequence_length])
        Y_sequences.append(window_labels[0])

X_seq = np.array(X_sequences)  # Shape: (Num_Clips, 10, 63)
Y_seq = np.array(Y_sequences)  # Shape: (Num_Clips,)

# --- 2. THREE-WAY STRATIFIED SPLIT (Train, Validation, Test) ---
# Step A: 70% Train, 30% Temp
X_train, X_temp, Y_train, Y_temp = train_test_split(
    X_seq, Y_seq, test_size=0.30, stratify=Y_seq, random_state=42
)

# Step B: Divide the 30% Temp evenly into Validation (15%) and Test (15%)
X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp, test_size=0.50, stratify=Y_temp, random_state=42
)


In [ ]:
# --- 3. FLATTEN 3D TO 2D & SAVE TO CSV ---
def save_lstm_to_csv(X_data, Y_data, filename):
    # Reshape from (Samples, 10, 63) to a flat 2D (Samples, 630) table
    num_samples = X_data.shape[0]
    X_flattened = X_data.reshape(num_samples, -1) 
    
    # Combine flattened landmarks and labels into a single DataFrame
    df_save = pd.DataFrame(X_flattened)
    df_save['label'] = Y_data
    
    # Save file
    df_save.to_csv(filename, index=False)
    print(f"Saved {filename} with shape: {df_save.shape}")

# Run the saving function for all 3 splits
save_lstm_to_csv(X_train, Y_train, config.LSTM_TRAIN_CSV)
save_lstm_to_csv(X_val, Y_val, config.LSTM_VALIDATION_CSV)
save_lstm_to_csv(X_test, Y_test, config.LSTM_TEST_CSV)

Saved d:\computer_vision_WP\Gesture Volume Control\DataSet\processed\lstm_train.csv with shape: (98403, 631)
Saved d:\computer_vision_WP\Gesture Volume Control\DataSet\processed\lstm_validation.csv with shape: (21086, 631)
Saved d:\computer_vision_WP\Gesture Volume Control\DataSet\processed\lstm_test.csv with shape: (21087, 631)
